# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

RDD joins are slow, so this notebook starts with a reproducible 5% sample for development

In [6]:
# Development mode
# workingCitations = rddCitations.sample(False, 0.05, 4253)
# workingPatents = rddPatents.sample(False, 0.05, 4253)

# Final mode: 
workingCitations = rddCitations
workingPatents = rddPatents

The RDD input is text, including quoted CSV headers. `csv.reader` handles quoted states and empty fields correctly. I retain the original patent row for the final augmented output, while `patent_states` is the smaller `(patent, state)` lookup used by the joins.

In [7]:
import csv

def parse_csv_line(line):
    return next(csv.reader([line]))

citations = (
    workingCitations
    .map(parse_csv_line)
    .filter(lambda row: row[0] != "CITING")
    .map(lambda row: (int(row[0]), int(row[1])))
    .cache()
)

parsed_patents = (
    workingPatents
    .map(parse_csv_line)
    .filter(lambda row: row[0] != "PATENT")
    .map(lambda row: (int(row[0]), row))
    .cache()
)

patent_states = (
    parsed_patents
    .filter(lambda item: item[1][4] == "US" and item[1][5] != "")
    .map(lambda item: (item[0], item[1][5]))
    .reduceByKey(lambda left, right: left)
    .cache()
)

patent_states.take(5)

[(3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA'),
 (3070806, 'PA')]

The first join keys citations by the citing patent. I then re-key by the cited patent for the second join. The result is keyed by citing patent and has `(cited_state, citing_state)` as its value. Missing patent records or missing states naturally disappear during these inner joins.

In [8]:
citation_by_citing = citations.map(lambda pair: (pair[0], pair[1]))
citing_with_state = citation_by_citing.join(patent_states)

citation_by_cited = citing_with_state.map(
    lambda item: (item[1][0], (item[0], item[1][1]))
)

citations_with_states = citation_by_cited.join(patent_states).map(
    lambda item: (item[1][0][0], (item[1][1], item[1][0][1]))
).cache()

citations_with_states.take(5)

[(5458680, ('PA', 'GA')),
 (5298066, ('PA', 'GA')),
 (4732748, ('PA', 'CA')),
 (5547574, ('PA', 'GA')),
 (5676748, ('PA', 'WA'))]

A joined record is a self-state citation when its cited and citing states are equal. Mapping matches to `(citing_patent, 1)` and reducing by key produces the required per-patent count.

In [9]:
same_state_counts = (
    citations_with_states
    .filter(lambda item: item[1][0] == item[1][1])
    .map(lambda item: (item[0], 1))
    .reduceByKey(lambda left, right: left + right)
    .cache()
)

same_state_counts.takeOrdered(5, key=lambda item: (-item[1], item[0]))

[(5959466, 125), (5983822, 103), (6008204, 100), (5952345, 98), (5958954, 96)]

A left outer join retains US patents with a known state even when their count is absent. Those missing values become zero in the appended final column. The top-ten display is sorted by count descending, then patent number ascending to break ties consistently.

In [10]:
augmented_patents = (
    parsed_patents
    .filter(lambda item: item[1][4] == "US" and item[1][5] != "")
    .leftOuterJoin(same_state_counts)
    .mapValues(lambda value: value[0] + [value[1] if value[1] is not None else 0])
)

top_10 = same_state_counts.takeOrdered(10, key=lambda item: (-item[1], item[0]))
top_10

[(5959466, 125),
 (5983822, 103),
 (6008204, 100),
 (5952345, 98),
 (5958954, 96),
 (5998655, 96),
 (5936426, 94),
 (5739256, 90),
 (5913855, 90),
 (5925042, 90)]

This assertion is expected to pass only after switching to the full-data assignments above and rerunning from the parsing cell. It checks the handout's example for patent 6009554.

In [12]:
patent_6009554 = same_state_counts.filter(
    lambda pair: pair[0] == 6009554
).take(1)

assert patent_6009554 and patent_6009554[0][1] == 8
print("Full-data check passed: patent 6009554 has 8 same-state citations.")

Full-data check passed: patent 6009554 has 8 same-state citations.
